# За кулисами теории вероятностей: ТДЗ - Геометрия теории вероятности 

The methods, examples and ideas for the present ipnyb are taken from  Демешев Борис Борисович [За кулисами теории вероятностей](https://www.hse.ru/edu/courses/1103572233)


Семен Цинман и Даниэль Фрейре 

### Инструкции использования

MiKTeX

FFmpeg

In [4]:

#you will need a latex engine, i used MiKTeX, which can be found here: 
# https://miktex.org/

# using your command promot, or powershell (windows) install the video complier 
# winget install ffmpeg

### Метод первого шага

Главная идея метода первого шага состоит в том чтобы вместо того чтобы пытаться просчитать все возможные пути развития событий до самого конца (что часто невозможно, так как игра может длиться бесконечно), мы фокусируемся только на первом шаге, от чего и исходит название метода. 

Мы применяем формулу полной вероятности или полного математического ожидания, разбивая сложную задачу на сумму более простых сценариев, зависящих от того, что произойдет на старте. Это позволяет выразить искомую величину (например, ожидаемый выигрыш) через саму себя, получив систему линейных уравнений.

**Когда стоит его применять:**
* Процесс имеет рекурсивную природу (состояния могут повторяться).
* Игра может длиться неограниченное количество времени (бесконечные марковские цепи).
* Необходимо найти математическое ожидание времени до поглощения (конца игры) или вероятность достижения определенного состояния.


Рассмотрим несколько наглядных примеров.

#### Кубик и шляпа

**Задача:**

 Игрок бросает правильный шестигранный кубик. В игре есть шляпа, в которой на данный момент находится $S$ рублей. В зависимости от броска (с вероятностью 1/6 для каждой грани), происходит одно из событий:
1. Прибавляем 1 к банку (переход в $S+1$) и продолжаем.
2. Прибавляем 2 к банку (переход в $S+2$) и продолжаем.
3. Прибавляем 3 к банку (переход в $S+3$) и продолжаем.
4. Игра заканчивается с выигрышем S рублей.
5. Деньги сгорают, банк обнуляется (переход в 0), продолжаем игру.
6. Конец игры с выплатой 0 рублей.

**Решение:**

Пусть $v(S)$ это ожидаемая цена позиции (наш средний выигрыш), если сейчас в шляпе $S$ рублей. Обуславливаясь первым броском, запишем математическое ожидание:
$$v(S) = \frac{1}{6}v(S+1) + \frac{1}{6}v(S+2) + \frac{1}{6}v(S+3) + \frac{1}{6}\cdot S + \frac{1}{6}v(0) + \frac{1}{6}\cdot 0$$

Поскольку переходы линейны, мы ищем решение в виде $v(S) = \alpha + \beta S$. Подставляя это предположение в уравнение и приравнивая коэффициенты при $S$ и свободные члены, мы получаем:
$$v(S) = \frac{5}{3} + \frac{1}{3}S$$
Это означает, что базовая ценность игры равна $\frac{5}{3}$, а каждый рубль в шляпе увеличивает нашу ожидаемую ценность на $\frac{1}{3}$.

In [ ]:
class DiceFirstStep(Scene):
    def construct(self):
        title = Text("Метод первого шага: Игра с кубиком", font_size=36).to_edge(UP)
        self.play(Write(title))

        vertices = ["S", "S+1", "S+2", "S+3", "Win(5)", "Lose(0)", "End(0)"]
        edges = [
            ("S", "S+1"), ("S", "S+2"), ("S", "S+3"),
            ("S", "Win(5)"), ("S", "Lose(0)"), ("S", "End(0)")
        ]
        
        g1 = Graph(vertices, edges, layout="spring", layout_scale=3,
                   labels=True, vertex_config={"radius": 0.4, "color": BLUE})
        
        self.play(Create(g1), run_time=2)
        
        # Подсветка первого шага (расщепление исходов)
        self.play(g1.vertices["S"].animate.set_color(YELLOW), run_time=0.5)
        
        # Анимация "растекания" вероятностей по граням
        for edge in edges:
            self.play(g1.edges[edge].animate.set_color(YELLOW), run_time=0.2)
            
        self.wait(2)

#### Геометрия случайных величин 

#### Задача 2: Ожидание комбинаций Орла (H) и Решки (T)

**Описание:** Подбрасывается правильная монета. Два игрока заключили пари: Игрок А ждет комбинацию `HTT`, а Игрок B ждет `TTH`. Тот, чья комбинация выпадет раньше, побеждает. Какова вероятность победы Игрока А?

**Решение:**
В этой игре состояние определяется последними выпавшими монетами. 
Если мы нарисуем граф переходов, то заметим ключевую логику: как только выпадает первый `H` (Орел), Игрок B (`TTH`) теряет шансы на победу. Для победы ему нужны две решки подряд `TT`, но любая решка после `H` начнет формировать комбинацию `HTT` для Игрока А. 

Следовательно, чтобы Игрок B выиграл, игра обязательно должна начаться с `TT`. Поскольку броски независимы, вероятность того, что первые два броска будут решками, равна 1/4. Значит, Игрок B выигрывает с вероятностью 1/4. 
Отсюда вероятность победы Игрока А (`HTT`) составляет:
$$P(X_{HTT} < X_{TTH}) = 1 - \frac{1}{4} = \frac{3}{4}$$

In [8]:
%%manim -qm -v WARNING DynamicProbabilityGeometry

from manim import *
import numpy as np

class DynamicProbabilityGeometry(Scene):
    def construct(self):
        # 1. Define Parameters
        sigma_x = 4.0
        sigma_y = 3.0
        
        # ValueTracker allows us to animate a variable over time
        corr_tracker = ValueTracker(1.0)
        
        # 2. Text and Formulas
        title = Text("Dynamic Correlation & Variance", font_size=36).to_edge(UP)
        
        def get_formula_text():
            corr = corr_tracker.get_value()
            var_sum = sigma_x**2 + sigma_y**2 + 2 * corr * sigma_x * sigma_y
            return MathTex(
                f"Var(X+Y) = {sigma_x**2:.1f} + {sigma_y**2:.1f} + 2({corr:.2f})({sigma_x})({sigma_y}) = {var_sum:.1f}"
            ).scale(0.7).next_to(title, DOWN)
        
        formula = always_redraw(get_formula_text)
        
        # 3. Vector Math & Updaters
        origin = DOWN * 2 + LEFT * 2
        
        vec_x = Arrow(origin, origin + RIGHT * sigma_x, buff=0, color=BLUE)
        label_x = MathTex(r"\vec{X}").next_to(vec_x, DOWN).set_color(BLUE)
        
        def get_vec_y():
            corr = np.clip(corr_tracker.get_value(), -1.0, 1.0)
            theta = np.arccos(corr)
            end_y = origin + np.array([sigma_y * np.cos(theta), sigma_y * np.sin(theta), 0])
            return Arrow(origin, end_y, buff=0, color=GREEN)
        
        vec_y = always_redraw(get_vec_y)
        
        def get_vec_sum():
            corr = np.clip(corr_tracker.get_value(), -1.0, 1.0)
            theta = np.arccos(corr)
            end_y = origin + np.array([sigma_y * np.cos(theta), sigma_y * np.sin(theta), 0])
            end_x = origin + RIGHT * sigma_x
            end_sum = end_x + (end_y - origin)
            return Arrow(origin, end_sum, buff=0, color=RED)
        
        vec_sum = always_redraw(get_vec_sum)
        
        def get_dashed_lines():
            corr = np.clip(corr_tracker.get_value(), -1.0, 1.0)
            theta = np.arccos(corr)
            end_y = origin + np.array([sigma_y * np.cos(theta), sigma_y * np.sin(theta), 0])
            end_x = origin + RIGHT * sigma_x
            end_sum = end_x + (end_y - origin)
            
            vg = VGroup()
            vg.add(DashedLine(end_x, end_sum, color=GREEN, dash_length=0.15))
            vg.add(DashedLine(end_y, end_sum, color=BLUE, dash_length=0.15))
            return vg
        
        lines = always_redraw(get_dashed_lines)
        
        # 4. Play Initial Setup
        self.play(Write(title), Write(formula))
        self.play(GrowArrow(vec_x), Write(label_x))
        self.play(GrowArrow(vec_y), GrowArrow(vec_sum), Create(lines))
        self.wait(1)
        
        # 5. The Dynamic Sweep Animation
        self.play(corr_tracker.animate.set_value(-1.0), run_time=4, rate_func=there_and_back)
        self.wait(0.5)
        
        self.play(corr_tracker.animate.set_value(0.0), run_time=2)
        self.wait(3)

Manim Community v0.20.1

In [12]:
from IPython.display import Video

# Укажите точный путь к вашему сгенерированному .mp4 файлу
video_path = "media/videos/ваше_имя_блокнота/720p30/DynamicProbabilityGeometry.mp4"

# embed=True зашивает видео прямо в блокнот (помогает обойти блокировки путей)
Video(video_path, embed=True, width=1000, html_attributes="controls autoplay loop")

### Метод траекторий  

### Аксиомы Гершеля Максвела 


### Мамикона (hard) 



### Критерий Келли (вроде это было про коней как раз если помню правильно)